# Denoising diffusion (DDPM)

A small U-Net is trained to predict the noise that was added to an image. Sampling starts from `x_T ~ N(0, I)` and removes that noise over `T = 1000` steps. That loop is why latency is far higher than a single GAN or VAE forward, which is the trilemma comparison in Q3. The saved trajectory is the bottom of Figure 5.

Run this on a Colab **GPU** runtime. The project folder must sit at `MyDrive/dl-sesi07`. Change `DRIVE_ROOT` in the setup cell if your folder name differs. Run the cells from top to bottom. Results are written to `artifacts/` on Drive. This notebook is the long session. If the runtime disconnects, run the training cell again; it resumes from the last epoch on Drive.


In [ ]:
import os
import sys
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/dl-sesi07"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path(DRIVE_ROOT)
else:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if "google.colab" in sys.modules:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


## Setup

Training images are scaled to `[-1, 1]`. The U-Net pads 28×28 to 32×32 so the two pooling steps stay even, then crops back.


In [ ]:
%matplotlib inline
import time

import torch
from torch.optim import Adam
from tqdm.auto import tqdm

from src.config import (
    DIFFUSION_STEPS,
    FID_SAMPLES,
    GRID_SAMPLES,
    SEED,
    get_device,
    set_seed,
    standard_image_noise,
    to_signed,
    to_unit,
)
from src.data import fixed_test_images, get_splits, make_loader
from src.io import artifact_dir, load_checkpoint, load_json, save_json, save_shared, save_torch
from src.metrics import measure_latency_ms, stability_from_curve
from src.models import DDPM
from src.plots import show_grid

set_seed(SEED)
DEVICE = get_device()
EPOCHS = 40
NAME = "ddpm"
KEEP_STEPS = [DIFFUSION_STEPS, 800, 600, 400, 200, 0]
print("device", DEVICE, "steps", DIFFUSION_STEPS)

train_set, _, _ = get_splits()
test_images, _ = fixed_test_images()
save_shared("test8.pt", {"images": test_images, "labels": []})


## Loss

For a random step `t`, the image is noised with the closed-form Gaussian `x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) eps`. The U-Net predicts `eps`, and the loss is MSE on that prediction. Adam learning rate is `2e-4`.


In [ ]:
model = DDPM().to(DEVICE)
optimizer = Adam(model.parameters(), lr=2e-4)
history = {"train_loss": [], "epoch_seconds": []}
start_epoch = 0
checkpoint = load_checkpoint(NAME, map_location=DEVICE)
if checkpoint is not None:
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    history = load_json(NAME, "history.json")
    start_epoch = checkpoint["epoch"] + 1
    print("resuming at epoch", start_epoch + 1)

for epoch in range(start_epoch, EPOCHS):
    set_seed(SEED + epoch)
    loader = make_loader(train_set, shuffle=True, seed=SEED + epoch)
    model.train()
    total = 0.0
    seen = 0
    start = time.perf_counter()
    for images, _ in tqdm(loader, leave=False):
        images = to_signed(images.to(DEVICE))
        loss = model.loss(images)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * images.size(0)
        seen += images.size(0)
    elapsed = time.perf_counter() - start
    history["train_loss"].append(total / seen)
    history["epoch_seconds"].append(elapsed)
    save_json(NAME, "history.json", history)
    save_torch(
        NAME,
        "checkpoint.pt",
        {"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict()},
    )
    print(f"epoch {epoch + 1:02d}  noise mse {total / seen:.4f}  {elapsed:.1f}s")


## Trajectory, grid, FID samples, latency

Latency is a full 64-image reverse chain, timed three times after one warmup. The 5,000 FID images are generated in batches and skipped if `samples_fid.pt` is already complete, so a disconnect during sampling can resume this cell.


In [ ]:
checkpoint = load_checkpoint(NAME, map_location=DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()
history = load_json(NAME, "history.json")

grid_noise = standard_image_noise(GRID_SAMPLES, SEED).to(DEVICE)
with torch.no_grad():
    frames = model.trajectory(grid_noise[:1].clone(), set(KEEP_STEPS))
    trajectory = torch.cat([to_unit(frames[step]) for step in KEEP_STEPS])
    grid = to_unit(model.sample(grid_noise.clone())).cpu()
save_torch(NAME, "trajectory.pt", {"images": trajectory, "steps": KEEP_STEPS})
save_torch(NAME, "samples_grid.pt", grid)

fid_path = artifact_dir(NAME) / "samples_fid.pt"
if fid_path.exists():
    fid_images = torch.load(fid_path, map_location="cpu", weights_only=False)
    print("loaded FID samples", tuple(fid_images.shape))
else:
    fid_chunks = []
    left = FID_SAMPLES
    seen = 0
    with torch.no_grad():
        while left:
            count = min(64, left)
            noise = torch.randn(count, 1, 28, 28, device=DEVICE)
            fid_chunks.append(to_unit(model.sample(noise)).cpu())
            left -= count
            seen += count
            print(f"sampled {seen}/{FID_SAMPLES}")
    fid_images = torch.cat(fid_chunks)

latency_noise = standard_image_noise(64, SEED).to(DEVICE)
latency = measure_latency_ms(lambda: model.sample(latency_noise.clone()), DEVICE, warmup=1, repeats=3)

save_torch(NAME, "samples_fid.pt", fid_images)
save_json(
    NAME,
    "metrics.json",
    {
        "latent_dim": "28x28",
        "sec_per_epoch": sum(history["epoch_seconds"]) / len(history["epoch_seconds"]),
        "latency_ms": latency,
        "test_mse": None,
        "test_psnr": None,
        "test_ssim": None,
        "stability": stability_from_curve(history["train_loss"]),
    },
)
show_grid(trajectory, nrow=6, title="Reverse chain, noise to image")
show_grid(grid, nrow=4, title="16 samples")
print("latency ms", round(latency, 1))
